In [ ]:
"""
Batch Tag Update Template
---------------------------------
Exemplo de atualização em lote via API REST
com controle, normalização e rastreabilidade.
"""

import re
import json
import time
import requests
import pandas as pd
from datetime import datetime

# ==============================
# CONFIGURAÇÕES
# ==============================

TOKEN = "Bearer SEU_TOKEN_AQUI"
API_GET = "https://api.exemplo.com/contacts?email="
API_PUT = "https://api.exemplo.com/contacts/{}/"
NOVA_TAG = "00000000-0000-0000-0000-000000000000"

EMAIL_COL = "email"
SLEEP_BETWEEN_REQUESTS_SEC = 0.2
TIMEOUT_SEC = 30

headers = {
    "Authorization": TOKEN,
    "Content-Type": "application/json"
}

# ==============================
# LOG
# ==============================

def log(msg: str):
    agora = datetime.now().strftime("%H:%M:%S")
    print(f"[{agora}] {msg}")

# ==============================
# VALIDAÇÃO
# ==============================

def is_valid_email(email: str) -> bool:
    if not isinstance(email, str):
        return False
    email = email.strip()
    return bool(re.match(r"^[^@\s]+@[^@\s]+\.[^@\s]+$", email))

# ==============================
# NORMALIZAÇÕES
# ==============================

def normalize_list(field):
    out = []
    for item in (field or []):
        if isinstance(item, dict) and "id" in item:
            out.append(str(item["id"]))
        elif isinstance(item, str):
            out.append(item)
    return list(dict.fromkeys(out))

def normalize_persona(persona):
    if isinstance(persona, dict) and "id" in persona:
        return str(persona["id"])
    if isinstance(persona, str):
        return persona
    return None

# ==============================
# BUILD PAYLOAD
# ==============================

def build_payload(contato, filtros):
    payload = {
        "first_name": contato.get("first_name"),
        "last_name": contato.get("last_name"),
        "email": contato.get("email"),
        "is_active": contato.get("is_active"),
        "filters": filtros
    }

    persona_uuid = normalize_persona(contato.get("persona"))
    if persona_uuid is not None:
        payload["persona"] = persona_uuid

    return payload

# ==============================
# PROCESSAMENTO EM LOTE
# ==============================

def processar_emails(df):
    sucessos, falhas = [], []

    emails_validos = [
        e.strip()
        for e in df[EMAIL_COL].astype(str)
        if is_valid_email(e)
    ]

    emails_unicos = list(dict.fromkeys(e.lower() for e in emails_validos))

    for email in emails_unicos:
        log(f"Processando {email}")

        r = requests.get(API_GET + email, headers=headers, timeout=TIMEOUT_SEC)

        if r.status_code != 200:
            falhas.append({"email": email, "erro": r.text})
            continue

        dados = r.json()
        if not dados.get("results"):
            continue

        contato = dados["results"][0]
        contato_id = contato["id"]

        filtros = normalize_list(contato.get("filters"))

        if NOVA_TAG not in filtros:
            filtros.append(NOVA_TAG)

        payload = build_payload(contato, filtros)

        resp = requests.put(
            API_PUT.format(contato_id),
            headers=headers,
            data=json.dumps(payload),
            timeout=TIMEOUT_SEC
        )

        if resp.status_code == 200:
            sucessos.append(email)
        else:
            falhas.append({"email": email, "erro": resp.text})

        time.sleep(SLEEP_BETWEEN_REQUESTS_SEC)

    return sucessos, falhas